# Web Scraping - A Complete, Hands-On Guide (Book Catalogue)

### CodeAlpha Internship · Task 1: Web Scraping

---

Objective: Build a complete, production-style web scraper from the
ground up - covering everything from inspecting a site's HTML structure
to robust, polite, error-tolerant data collection at scale.

Data source: [Books to Scrape](https://books.toscrape.com) - a public
sandbox website built specifically for practicing web scraping. It
requires no authentication and explicitly permits scraping, making it a
safe, legal target for this project.

Tech stack: Python · Requests · BeautifulSoup · Pandas


## 1. Import Libraries

In [107]:
from __future__ import annotations

import time
import json
import logging
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pandas as pd

# Simple, readable logging instead of scattered print statements
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                     datefmt="%H:%M:%S")
logger = logging.getLogger("scraper")


## 2. Checking robots.txt (Scraping Ethics)

Before scraping any site, it's good practice to check its robots.txt -
a file that states which parts of the site are (and aren't) allowed to be
crawled by bots. Doing this programmatically, rather than skipping it, is
part of writing a responsible scraper.


In [108]:
SITE_ROOT = "https://books.toscrape.com/"
ROBOTS_URL = urljoin(SITE_ROOT, "robots.txt")

robot_parser = RobotFileParser()
robot_parser.set_url(ROBOTS_URL)

try:
    robot_parser.read()
    can_fetch = robot_parser.can_fetch("*", SITE_ROOT)
    logger.info(f"robots.txt found at {ROBOTS_URL}")
    logger.info(f"Allowed to scrape '{SITE_ROOT}'? {can_fetch}")
except Exception as e:
    logger.warning(f"Could not read robots.txt ({e}). Proceeding cautiously with a slow crawl rate.")


Note: books.toscrape.com is a sandbox site built for scraping
practice, so it places no restrictions on crawling - but running this
check is a habit worth keeping for every real scraping project.


## 3. Inspecting the Page Structure

Before writing any parsing logic, it helps to fetch one page and actually
look at its HTML - this tells us which tags and CSS classes hold the data
we want. We'll fetch the first catalogue page and print a snippet of its
raw HTML to see the structure we're working with.


In [109]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

sample_response = requests.get("https://books.toscrape.com/catalogue/page-1.html",
                                headers=HEADERS, timeout=10)
sample_soup = BeautifulSoup(sample_response.text, "html.parser")


In [110]:
# Look at the HTML for just the first book card on the page
first_card = sample_soup.select_one("article.product_pod")
print(first_card.prettify()[:1200])

<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



What we can see from this snippet:
- Each book is wrapped in <article class="product_pod">
- The title lives in h3 > a, in the title attribute (more reliable
  than the visible text, which can be truncated with "...")
- The price is in p.price_color
- The star rating is encoded as a CSS class on p.star-rating
  (e.g. star-rating Three) rather than as visible text — a common
  pattern worth knowing how to handle
- Availability text lives in p.availability

This kind of inspection - either by printing HTML like above, or using
your browser's "Inspect Element" tool - is the first real step of any
scraping project, before a single line of parsing logic is written.


## 4. Building a Robust Request Function

A production-quality scraper shouldn't just call requests.get() and
hope for the best. This section builds a request function with:

- Automatic retries with exponential backoff for transient failures
  (e.g. HTTP 500, connection drops)
- A shared Session for connection reuse (faster, more efficient
  than opening a new connection per request)
- A timeout, so a hung server never freezes the whole scrape
- Clear error logging when a page truly fails


In [111]:
def build_session(total_retries: int = 3, backoff_factor: float = 0.5) -> requests.Session:
    """Create a requests Session configured with automatic retries."""
    session = requests.Session()
    session.headers.update(HEADERS)

    retry_strategy = Retry(
        total=total_retries,
        backoff_factor=backoff_factor,          # wait 0.5s, 1s, 2s, ... between retries
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

In [112]:
SESSION = build_session()

In [113]:
def fetch_page(url: str, timeout: int = 10) -> BeautifulSoup | None:
    """Fetch a URL and return parsed HTML, or None if the request ultimately fails."""
    try:
        response = SESSION.get(url, timeout=timeout)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to fetch {url}: {e}")
        return None

## 5. Parsing a Single Listing Page

Now we extract structured data from one page. Each field is pulled
defensively - using .get()-style safe lookups and try/except
where a missing element shouldn't crash the whole scrape, just that one
field.


In [114]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def parse_rating(card) -> int | None:
    """Star ratings are encoded as a CSS class, e.g. class="star-rating Three"."""
    tag = card.select_one("p.star-rating")
    if not tag:
        return None
    classes = tag.get("class", [])
    word = next((c for c in classes if c != "star-rating"), None)
    return RATING_WORDS.get(word)


In [115]:
def parse_book_card(card, category: str) -> dict:
    """Extract one book's fields from its listing-page card. Never raises --
    missing fields are recorded as None so one bad record doesn't kill the run."""

    def safe_text(selector, clean=lambda t: t):
        el = card.select_one(selector)
        return clean(el.get_text(strip=True)) if el else None

    title_tag = card.select_one("h3 a")
    title = title_tag["title"].strip() if title_tag and title_tag.has_attr("title") else None

    price = safe_text(".price_color", clean=lambda t: t.replace("£", "").replace("Â", "").strip())

    return {
        "title": title,
        "price": price,
        "rating": parse_rating(card),
        "availability": safe_text(".availability"),
        "category": category,
        "product_url": urljoin(BASE_URL, title_tag["href"]) if title_tag else None,
    }

In [116]:
BASE_URL = "https://books.toscrape.com/catalogue/"

# Quick test on the page we already fetched in Section 3
test_cards = sample_soup.select("article.product_pod")
print(f"Found {len(test_cards)} book cards on this page\n")

sample_parsed = [parse_book_card(c, category="Unknown") for c in test_cards[:3]]
pd.DataFrame(sample_parsed)


Found 20 book cards on this page



,title,price,rating,availability,category,product_url
0,A Light in the Attic,51.77,3,In stock,Unknown,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,Unknown,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,Unknown,https://books.toscrape.com/catalogue/soumissio...


## 6. Extracting Book Details - Nested Scraping

The genre/category isn't shown on the listing page - it only appears on
each book's own product page, in the breadcrumb navigation. This means
our scraper has to make a second request per book: one classic
pattern in real-world scraping, where the data you need is spread across
multiple pages.


In [117]:
def get_category(product_url: str) -> str:
    """Visit a book's product page and read its category from the breadcrumb."""
    soup = fetch_page(product_url)
    if soup is None:
        return "Unknown"

    breadcrumb = soup.select("ul.breadcrumb li a")
    # breadcrumb[0] = "Home", breadcrumb[1] = category, breadcrumb[2] would be the book title
    if len(breadcrumb) >= 2:
        return breadcrumb[1].get_text(strip=True)
    return "Unknown"


In [118]:
# Test it on the first book we found above
first_title_tag = test_cards[0].select_one("h3 a")
first_book_url = urljoin(BASE_URL, first_title_tag["href"])

print("Product page URL:", first_book_url)
print("Category found:  ", get_category(first_book_url))


Product page URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Category found:   Books


## 7. Handling Pagination

The catalogue spans 50 pages (20 books each). Rather than hardcoding page
numbers, we follow the site's own "next" link - a more robust pattern
that keeps working even if the site's URL scheme or page count changes.


In [119]:
def get_next_page_url(soup: BeautifulSoup, current_url: str) -> str | None:
    """Return the absolute URL of the next listing page, or None if there isn't one."""
    next_link = soup.select_one("li.next a")
    if next_link:
        return urljoin(current_url, next_link["href"])
    return None


## 8 Putting It All Together - The Full Scraper

This function combines everything above into one polite, resilient
crawl loop:

- Follows pagination automatically
- Makes a nested request per book for its category
- Waits briefly between requests (REQUEST_DELAY) so we don't hammer the
  server
- Logs progress as it goes, and keeps going even if a handful of
  individual books fail


In [120]:
START_URL = "https://books.toscrape.com/catalogue/page-1.html"
REQUEST_DELAY = 0.2  # seconds between requests -- politeness matters

In [121]:
def scrape_catalogue(max_pages: int | None = 5) -> list[dict]:
    """
    Crawl the paginated book catalogue and return a list of book records.
    max_pages caps how many listing pages to visit (~20 books/page);
    pass None to scrape the entire ~1,000-book catalogue.
    """
    all_records = []
    url = START_URL
    page_num = 1
    failed_books = 0

    while url:
        logger.info(f"Fetching listing page {page_num}: {url}")
        soup = fetch_page(url)
        if soup is None:
            logger.warning(f"Skipping page {page_num} -- failed to fetch after retries")
            break

        cards = soup.select("article.product_pod")
        for card in cards:
            title_tag = card.select_one("h3 a")
            if not title_tag:
                failed_books += 1
                continue

            product_url = urljoin(BASE_URL, title_tag["href"])
            category = get_category(product_url)
            record = parse_book_card(card, category)
            all_records.append(record)
            time.sleep(REQUEST_DELAY)

        logger.info(f"  -> {len(cards)} books parsed (running total: {len(all_records)})")

        if max_pages is not None and page_num >= max_pages:
            break

        next_url = get_next_page_url(soup, url)
        if next_url is None:
            break

        url = next_url
        page_num += 1

    if failed_books:
        logger.warning(f"{failed_books} book cards could not be parsed and were skipped")

    logger.info(f"Done. Scraped {len(all_records)} books across {page_num} page(s).")
    return all_records


## 9. Running the Scraper

By default this scrapes 5 listing pages (~100 books) to keep runtime
short and friendly for a demo. Set max_pages=None below to crawl the
entire catalogue (1,000 books) - expect that to take several minutes,
since it makes one extra request per book for the category.


In [103]:
records = scrape_catalogue(max_pages=5)

df = pd.DataFrame(records)
print(f"\nScraped {len(df)} books, {df.shape[1]} fields each")
df.head(10)



Scraped 100 books, 6 fields each


,title,price,rating,availability,category,product_url
0,A Light in the Attic,51.77,3,In stock,Books,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,Books,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,Books,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,Books,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,Books,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,1,In stock,Books,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,Books,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,Books,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,Books,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,1,In stock,Books,https://books.toscrape.com/catalogue/the-black...


## 10. Saving the Data (CSV & JSON)

We export in two formats: CSV for easy use in Pandas/Excel, and
JSON for use in web apps or downstream pipelines that prefer
structured records over flat tables.


In [104]:
import os

os.makedirs("data", exist_ok=True)

csv_path = "data/raw_data.csv"
json_path = "data/raw_data.json"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)

print(f"Saved {len(df)} records to:")
print(f"  - {csv_path}")
print(f"  - {json_path}")


Saved 100 records to:
  - data/raw_data.csv
  - data/raw_data.json


## 11. Sanity-Checking the Scraped Data

This is not data cleaning - just quick checks to confirm the scrape
itself actually worked as intended, before handing the raw data off to a
separate cleaning/analysis step.


In [105]:
print("Shape:", df.shape)
print()
print("Missing values per field:")
print(df.isnull().sum())
print()
print("Unique categories found:", df["category"].nunique())
print("Unique ratings found:", sorted(df["rating"].dropna().unique().tolist()))


Shape: (100, 6)

Missing values per field:
title           0
price           0
rating          0
availability    0
category        0
product_url     0
dtype: int64

Unique categories found: 1
Unique ratings found: [1, 2, 3, 4, 5]


In [106]:
# Spot-check a few random rows to eyeball data quality
df.sample(min(5, len(df)), random_state=42)


,title,price,rating,availability,category,product_url
83,"Political Suicide: Missteps, Peccadilloes, Bad...",36.28,2,In stock,Books,https://books.toscrape.com/catalogue/political...
53,This One Summer,19.49,4,In stock,Books,https://books.toscrape.com/catalogue/this-one-...
70,The Art Forger,40.76,3,In stock,Books,https://books.toscrape.com/catalogue/the-art-f...
45,When We Collided,31.77,1,In stock,Books,https://books.toscrape.com/catalogue/when-we-c...
44,Without Borders (Wanderlove #1),45.07,2,In stock,Books,https://books.toscrape.com/catalogue/without-b...


## 12. Final

Things to keep in mind for any real-world scraping project:
- Always re-check a site's Terms of Service, not just robots.txt -
  some sites prohibit scraping even where robots.txt is permissive.
- Scraping HTML is inherently fragile - if a site redesigns its markup,
  CSS selectors will need updating. Keep parsing logic isolated in small,
  well-named functions (like above) so fixes stay localized.
- For JavaScript-heavy sites (where content loads dynamically), a tool
  like Selenium or Playwright is often needed instead of requests +
  BeautifulSoup, which only see the initial HTML response.
- This notebook intentionally stops at raw, validated data - cleaning,
  exploratory analysis, and visualization belong in a separate, focused
  notebook.
